[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/humanization/humanization_guide/08_structure/08_structure_lab.ipynb)


# 08 — 구조 검증 — IgFold 로 직접 접고 6개 CDR RMSD 비교

> 본문 [`08_structure.md`](08_structure.md) 와 **한 절씩 짝지어** 보세요.
> **실측 소요 —** IgFold VH+VL 폴딩 **7.1초**(parental) · **12.0초**(humanized)
> **앞 랩에서 이어져요** — Ch.05 Sapiens 결과 의 `my_run/` 을 먼저 찾고, 없으면 커밋된 `data/` 로 대신합니다.

## 0) Colab 준비 — 저장소 클론 & 작업 폴더 이동

이 셀이 저장소를 클론하고 `08_structure/` 로 이동해 필요한 패키지만 깝니다(로컬에서 `08_structure/` 안에 열었다면 클론 생략).
끝나면 **`my_run/`**(내가 만들 결과)과 **`data/`**(커밋된 레퍼런스)가 준비돼요 — 랩은 `my_run/` 을 먼저 찾고 없으면 `data/` 로 폴백하며 어느 쪽을 썼는지 매번 print 합니다.

In [ ]:
# ====== Colab/로컬 공용 부트스트랩 (모든 챕터 공통) ======
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # 이 가이드 저장소 (fork 했다면 본인 주소로 바꾸세요)
CLONE_AS = "bio-guides"
CHAPTER  = "08_structure"
APT_PKGS = ""     # Colab 에서만: 시스템 패키지 (hmmer = ANARCI 가 부르는 hmmscan)
PIP_PKGS = "pandas matplotlib biopython py3Dmol gemmi"     # 없는 것만 설치

import os, sys, json, time, shutil, pathlib, subprocess, importlib, importlib.util
IN_COLAB = "google.colab" in sys.modules

# HuggingFace 가중치 다운로드가 '멈춘 채' 매달리는 일을 막아요.
# (멈춤은 예외가 안 나서 폴백이 안 걸려요 — 타임아웃을 걸어 실패로 바꿔야 data/ 로 이어져요)
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "30")   # 스트림 30초 무응답 → 끊고 재시도
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "15")

def _run(cmd, check=True, timeout=None, quiet=False):
    # timeout 을 꼭 주세요 — 네트워크가 '멈춘 채' 매달리면 예외가 안 나서 data/ 폴백이 안 걸립니다.
    """quiet=True 면 출력을 삼키고 **실패했을 때만** 보여줘요.
    apt-get 은 "(Reading database ... 5%(Reading database ... 10%" 같은 진행률을 600자 넘게 쏟아내는데,
    그게 노트북을 연 학습자가 보는 첫 화면을 덮어버려서 실패로 오해하게 만들거든요."""
    print("$", cmd)
    if not quiet:
        return subprocess.run(cmd, shell=True, check=check, timeout=timeout)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    if p.returncode != 0:
        print((p.stdout or "") + (p.stderr or ""))
        if check:
            raise subprocess.CalledProcessError(p.returncode, cmd)
    return p

_MARK = "humanization_viz.py"          # 이 파일이 있는 폴더 = 가이드 루트

def _find_root():
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    # 클론 직후: cwd '아래'만 깊이 3까지 훑는다.
    # (상위로 올라가 rglob 하면 Colab 에서는 / 전체를 뒤지게 된다 — 매우 느리고 엉뚱한 경로를 잡는다)
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "가이드 루트를 못 찾았어요. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

GUIDE_ROOT = ROOT.resolve()
os.chdir(GUIDE_ROOT / CHAPTER)         # data/ 상대경로가 그대로 동작
sys.path.insert(0, str(GUIDE_ROOT))    # humanization_viz import
sys.path.insert(0, str(GUIDE_ROOT / CHAPTER))

_IMPORT_NAME = {"biopython": "Bio", "pyyaml": "yaml", "scikit-learn": "sklearn"}

def _have(mod):
    try:
        return importlib.util.find_spec(mod) is not None
    except Exception:
        return False

def _ensure(pkgs):
    miss = [p for p in pkgs.split() if not _have(_IMPORT_NAME.get(p, p.replace("-", "_")))]
    if miss:
        print("설치:", " ".join(miss))
        _run(f'"{sys.executable}" -m pip -q install ' + " ".join(miss))
        importlib.invalidate_caches()

_APT = APT_PKGS.split() if (APT_PKGS and IN_COLAB) else []   # 모아서 apt 를 한 번만 돌립니다
if PIP_PKGS:
    _ensure(PIP_PKGS)

def _ensure_pkg_resources():
    # setuptools 81+(2026-02) 이 pkg_resources 를 없앴는데 IgFold 의존성이 이걸 import 합니다.
    if importlib.util.find_spec("pkg_resources") is None:
        _run(f'"{sys.executable}" -m pip -q install "setuptools<81"')
        importlib.invalidate_caches()

import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    _APT.append("fonts-nanum")             # Colab 엔 한글 폰트가 없어 라벨이 □ 로 깨져요

if _APT:                                   # apt 인덱스 갱신은 한 번만 (ANARCI 는 hmmscan 이 필요해요)
    _run("apt-get -qq update", timeout=600, quiet=True)
    _run("DEBIAN_FRONTEND=noninteractive apt-get -qq install -y " + " ".join(_APT), timeout=900, quiet=True)


# ── ① 내가 만든 결과(my_run) 우선 · ② 없으면 커밋된 레퍼런스(data/) ─────────────
MY  = pathlib.Path("my_run"); MY.mkdir(exist_ok=True)
REF = pathlib.Path("data")

def find_one(pattern, ref=REF, quiet=False):
    """산출물 하나 찾기 — 내가 만든 my_run/ 을 먼저 뒤지고, 없으면 커밋된 레퍼런스에서."""
    hits = sorted(MY.rglob(pattern))
    if hits:
        if not quiet: print(f"[내 결과]   {hits[0]}")
        return hits[0]
    hits = sorted(pathlib.Path(ref).glob(pattern))
    assert hits, f"'{pattern}' 을 my_run/ 에서도 {ref}/ 에서도 찾지 못했어요."
    if not quiet: print(f"[레퍼런스] {hits[0]}")
    return hits[0]

def find_prev(chapter, pattern, ref=REF, quiet=False):
    """앞 챕터에서 직접 만든 산출물 → 없으면 이 챕터 data/ 레퍼런스."""
    hits = sorted((GUIDE_ROOT / chapter / "my_run").rglob(pattern))
    if hits:
        if not quiet: print(f"[내 결과 · {chapter}] {hits[0]}")
        return hits[0]
    return find_one(pattern, ref, quiet)

def read_fasta(path):
    out, name, buf = {}, None, []
    for line in pathlib.Path(path).read_text().splitlines():
        if line.startswith(">"):
            if name: out[name] = "".join(buf)
            name, buf = line[1:].strip(), []
        elif line.strip():
            buf.append(line.strip())
    if name: out[name] = "".join(buf)
    return out

def write_fasta(path, records):
    pathlib.Path(path).write_text("".join(f">{k}\n{v}\n" for k, v in records.items()))
    return pathlib.Path(path)

PARENTAL = read_fasta(REF / "parental.fasta")      # 가이드 전체를 관통하는 러닝 예제
VH = PARENTAL["parental_H"]
VL = PARENTAL["parental_L"]

print("가이드 루트 :", GUIDE_ROOT)
print("작업 폴더   :", pathlib.Path.cwd())
print(f"러닝 예제   : VH {len(VH)} aa · VL {len(VL)} aa (mouse hybridoma 가정)")

## 1) 직접 실행 — parental·humanized 를 다시 접기 (본문 8.1)

서열 지표가 전부 좋아져도 **CDR loop 모양이 망가지면 결합력은 사라져요.** 그래서 서열을 만든 도구(Sapiens)가 아니라 **그 사정을 전혀 모르는 모델**에게 다시 접게 해요. 만드는 쪽과 검증하는 쪽을 분리하는 거예요.

IgFold(+AntiBERTy)는 VH+VL Fv 하나를 CPU 로 접어요. 기본값으로 부르면 두 번 죽으니 아래 두 옵션을 꺼요.

| 옵션 | 왜 끄나 |
|---|---|
| `do_refine=False` | `True` 면 **PyRosetta** 를 요구하고, 없으면 그 자리에서 `exit()` 합니다 |
| `do_renum=False` | `True` 면 abnumber 로 재numbering 하는데, 우리 VL 의 C-말단 `G` 가 IMGT 범위 밖이라 **AssertionError** 로 죽어요 |

스레드도 묶어요(`OMP_NUM_THREADS=4`) — 과부하 머신에서 IgFold forward 가 간헐적으로 SIGILL 로 죽는 걸 막아요(실측).

In [ ]:
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["CUDA_VISIBLE_DEVICES"] = ""     # AntiBERTy 가 부모의 try_gpu 를 무시하므로 여기서 차단

_ensure("igfold")
_ensure_pkg_resources()      # IgFold 의존성이 pkg_resources 를 import 합니다(setuptools 81+ 에서 제거됨)

# 후보 서열 — Ch.05 에서 내가 만든 가드 없는 argmax 결과 우선(가드 적용본을 쓰려면 파일명만 바꾸세요)
hp = GUIDE_ROOT / "05_humanize_sapiens" / "my_run" / "sapiens_humanized_noguard.fasta"
if hp.exists():
    print(f"[내 결과 · 05_humanize_sapiens] {hp}")
else:
    hp = find_one("sapiens_humanized.fasta")      # 출처 라벨은 find_one 이 찍어요
f = read_fasta(hp)
hum_h, hum_l = f["sapiens_humanized_VH"], f["sapiens_humanized_VL"]

targets = {"parental": {"H": VH, "L": VL},
           "sapiens_humanized": {"H": hum_h, "L": hum_l}}

mine = {}
try:
    from igfold import IgFoldRunner
    t0 = time.time(); runner = IgFoldRunner(); load_s = time.time() - t0
    for name, seqs in targets.items():
        t0 = time.time()
        runner.fold(str(MY / f"{name}.pdb"), sequences=seqs, do_refine=False, do_renum=False)
        mine[name] = time.time() - t0
        (MY / f"{name}_timing.json").write_text(json.dumps(
            {"load_time_sec": load_s, "fold_time_sec": mine[name], "prmsd_available": True}, indent=2))
        print(f"{name}: {mine[name]:.1f}초 → {MY / (name + '.pdb')}")
except Exception as e:
    print("IgFold 실행 실패:", type(e).__name__, str(e)[:200])
    print("→ 아래 분석은 레퍼런스 구조(data/parental.pdb · data/sapiens_humanized.pdb)로 그대로 이어져요.")

ref_t = {n: json.loads((REF / f"{n}_timing.json").read_text())["fold_time_sec"] for n in targets}
print("\n레퍼런스 실측 폴딩 —", " · ".join(f"{n} {v:.4f}초" for n, v in ref_t.items()))
if mine:
    print("내 폴딩       —", " · ".join(f"{n} {v:.1f}초" for n, v in mine.items()))

In [ ]:
import pandas as pd

display(pd.DataFrame([
    {"도구": "IgFold (+AntiBERTy)", "역할": "항체 Fv 구조 예측", "이 랩": "실행 — 이 챕터의 모든 수치"},
    {"도구": "ABodyBuilder3",       "역할": "항체 Fv 구조 예측", "이 랩": "미실행 — GPU·모델 가중치"},
    {"도구": "ImmuneBuilder",       "역할": "항체 Fv 구조 예측", "이 랩": "미실행 — GPU·모델 가중치"},
    {"도구": "AntiFold",            "역할": "inverse folding · residue tolerance", "이 랩": "미실행 — GPU"},
]))
print("아래 pRMSD·RMSD 는 전부 IgFold 한 도구에서 나온 값이에요.")
print("예측기가 바뀌면 절대값도 바뀌니, 다른 도구가 낸 수치와 나란히 놓고 비교하지 마세요.")
print("본문 8.3 의 AntiFold backmutation 우선순위도 이 환경에서는 돌리지 않았어요 — 명령 템플릿과 판정 기준만 있어요.")

## 2) 내 결과 확인 — 잔기별 예측 오차 (본문 8.2)

IgFold 는 PDB 의 B-factor 자리에 **잔기별 예측 오차(Å)** 를 적어요 — **낮을수록 확신**이에요. 겹쳐 보기 전에, 모델이 스스로 "여기는 자신 없다"고 말하는 자리부터 봐요.

그림은 `my_run/` 에 저장해요. 본문이 인용하는 `08_prmsd.png` 는 그대로 둡니다.

In [ ]:
import humanization_viz          # import 만으로 한글 폰트가 등록돼요(안 하면 제목·축이 □ 로 깨져요)
import matplotlib.pyplot as plt
from Bio.PDB import PDBParser
from IPython.display import Image, display

parser = PDBParser(QUIET=True)

def pick(df, *names):
    """산출물마다 컬럼명이 달라요(prmsd ↔ prmsd_ca_rmsd_angstrom_bfactor) — 있는 쪽을 골라요."""
    for n in names:
        if n in df.columns:
            return n
    raise KeyError(f"{names} 중 아무 컬럼도 없어요 — 실제 컬럼은 {list(df.columns)} 예요.")

def prmsd_table(pdb, name):
    """PDB → 잔기별 CA B-factor(= 예측 RMSD) 표. pos = 체인 안 1-based 번호."""
    rows = []
    for ch in parser.get_structure(name, str(pdb))[0]:
        i = 0
        for res in ch:
            if "CA" in res:
                i += 1
                rows.append({"chain": ch.id, "resnum": res.id[1], "pos": i,
                             "resname": res.get_resname(), "prmsd": float(res["CA"].get_bfactor())})
    return pd.DataFrame(rows)

pdbs = {n: find_one(f"{n}.pdb") for n in ("parental", "sapiens_humanized")}
tabs = {n: prmsd_table(p, n) for n, p in pdbs.items()}
for n, t in tabs.items():
    t.to_csv(MY / f"{n}_prmsd.csv", index=False)
    assert pick(t, "prmsd", "prmsd_ca_rmsd_angstrom_bfactor")

# CDR 좌표 — Ch.04 에서 만든 표 우선, 없으면 이 챕터 data/ (경로를 하드코딩하지 않아요)
ct = pd.read_csv(find_prev("04_sequence_qc", "cdr_table_imgt.csv", quiet=True))
CDR = {}
for _, r in ct.iterrows():
    seq = VH if r["chain"] == "H" else VL
    st = seq.find(r["sequence"]) + 1
    assert st > 0, f"CDR {r['chain']}-{r['cdr']} 를 parental 서열에서 못 찾았어요 — CDR 표와 parental.fasta 가 어긋나요."
    CDR[(r["chain"], r["cdr"])] = (st, st + len(r["sequence"]) - 1)
print("CDR 구간(체인 안 1-based) —", " · ".join(f"{c}-{n} {lo}~{hi}" for (c, n), (lo, hi) in CDR.items()))

fig, axes = plt.subplots(2, 1, figsize=(12, 7))
for ax, chain in zip(axes, ("H", "L")):
    for n, t in tabs.items():
        s = t[t.chain == chain]
        ax.plot(s["pos"], s[pick(s, "prmsd", "prmsd_ca_rmsd_angstrom_bfactor")], lw=1.6, label=n)
    for (c, nm), (lo, hi) in CDR.items():
        if c == chain:
            ax.axvspan(lo, hi, color="#c0508a", alpha=0.12)
    ax.set_ylabel("예측 RMSD (Å) ↓ 좋음")
    ax.set_title(f"IgFold 잔기별 예측 오차 — V{chain} (분홍 = CDR)", fontweight="bold")
    ax.grid(alpha=0.25); ax.legend()
axes[1].set_xlabel("체인 안 잔기 번호 (1-based)")
fig.tight_layout()
png = MY / "08_prmsd.png"; fig.savefig(png, dpi=150); plt.close(fig)
display(Image(str(png)))

cdr_pos = {ch: {i for (c, nm), (lo, hi) in CDR.items() if c == ch for i in range(lo, hi + 1)}
           for ch in ("H", "L")}
for n, t in tabs.items():
    col = pick(t, "prmsd", "prmsd_ca_rmsd_angstrom_bfactor")
    for chain in ("H", "L"):
        s = t[t.chain == chain]
        inc = s["pos"].isin(cdr_pos[chain])
        cdr_m, fr_m = s[inc][col].mean(), s[~inc][col].mean()
        print(f"  {n:18s} V{chain}  전체 {s[col].mean():.3f} Å · CDR {cdr_m:.3f} · framework {fr_m:.3f}"
              f"  → CDR 이 framework 의 {cdr_m / fr_m:.1f}배")
print("\n판정 — CDR 이 framework 보다 크게 나오는 게 정상이에요(loop 라서 원래 흔들려요).")
print("두 구조가 같은 패턴이면 예측 품질이 비슷하다는 뜻이라, 다음 절의 RMSD 를 서로 비교해도 돼요.")

## 3) 직접 실행 — framework 로 정렬한 뒤 CDR 별 RMSD (본문 8.2)

재는 순서가 중요해요. **framework CA 로 먼저 정렬**하고, 그 상태에서 CDR 만 RMSD 를 재요. 전체를 한꺼번에 정렬하면 loop 의 변화가 framework 오차에 묻혀 버려요.

짝은 **자리 번호**로 지어요. Sapiens 는 자리마다 잔기를 바꿔 쓸 뿐 넣거나 빼지 않아서, parental 의 i 번째와 humanized 의 i 번째가 곧 같은 자리거든요(Ch.05 의 `F101V` 같은 표기가 바로 이 번호예요). 길이가 달라지는 후보(Humatch 처럼 indel 이 난 경우)에서만 서열 정렬로 짝을 찾고, 짝이 없는 자리는 표에서 빠져요.

본문 8.2 표가 요구하는 건 CDR-H3 하나가 아니라 **6개 CDR 전부**예요.

In [ ]:
import difflib, numpy as np
from Bio.SeqUtils import seq1
from Bio.PDB import Superimposer

def ca_res(pdb, chain):
    """한 체인에서 CA 를 가진 잔기 목록(파일 순서 = 서열 순서)."""
    return [r for r in parser.get_structure("s", str(pdb))[0][chain] if "CA" in r]

def pair_index(a, b):
    """0-based 잔기 인덱스를 1:1 로 짝지어요.

    길이가 같으면 **자리끼리 그대로** 짝지어요 — Sapiens argmax 는 자리별 치환이라 indel 이 없고,
    Ch.05 의 mutation 표(F101V 처럼 자리 번호로 적힌 것)와 짝이 어긋나면 안 되거든요.
    (치환이 몰린 구간을 서열 정렬에 맡기면 '한 칸 밀린 삽입'으로 읽혀 엉뚱한 잔기끼리 재게 돼요.)
    길이가 다른 후보(Humatch 처럼 indel 이 난 경우)에서만 정렬로 짝을 찾고, 짝 없는 자리는 빠져요."""
    if len(a) == len(b):
        return {i: i for i in range(len(a))}
    m = {}
    for op, i1, i2, j1, j2 in difflib.SequenceMatcher(None, a, b, autojunk=False).get_opcodes():
        if op in ("equal", "replace"):
            for k in range(min(i2 - i1, j2 - j1)):
                m[i1 + k] = j1 + k
    return m

rows, head, notes, ALIGN = [], {}, [], {}      # ALIGN = 4절 3D 겹쳐보기가 그대로 재사용할 정렬 결과
for chain in ("H", "L"):
    P, Hm = ca_res(pdbs["parental"], chain), ca_res(pdbs["sapiens_humanized"], chain)
    # 서열은 FASTA 가 아니라 접힌 구조에서 직접 읽어요 — 폴백으로 구조만 레퍼런스가 되어도 어긋나지 않아요
    a = seq1("".join(r.get_resname() for r in P))
    b = seq1("".join(r.get_resname() for r in Hm))
    if a != (VH if chain == "H" else VL):
        notes.append(f"V{chain} — 접힌 parental 구조의 서열이 parental.fasta 와 달라요. 구조 쪽 서열을 기준으로 쟀어요.")
    pair = pair_index(a, b)
    spans = {nm: (lo, hi) for (c, nm), (lo, hi) in CDR.items() if c == chain}
    in_cdr = {i for lo, hi in spans.values() for i in range(lo - 1, hi)}
    fw = [i for i in sorted(pair) if i not in in_cdr]
    assert len(fw) > 20, f"V{chain} framework 짝이 {len(fw)}개뿐이에요 — CDR 표가 이 서열의 것이 맞는지 확인하세요."

    sup = Superimposer()
    sup.set_atoms([P[i]["CA"] for i in fw], [Hm[pair[i]]["CA"] for i in fw])
    sup.apply([at for r in Hm for at in r])          # humanized 를 framework 기준으로 옮겨 놓고 CDR 을 재요
    ALIGN[chain] = {"sup": sup, "pair": pair, "fw": fw, "spans": spans}   # 4절이 이 정렬을 그대로 씁니다

    def rmsd(idx):
        d = np.array([P[i]["CA"].coord - Hm[pair[i]]["CA"].coord for i in idx])
        return float(np.sqrt((d ** 2).sum() / len(d)))

    def muts(idx):
        return [f"{a[i]}{i + 1}{b[pair[i]]}" for i in idx if a[i] != b[pair[i]]]

    rows.append({"부위": f"V{chain} framework", "정렬 기준": "이 체인 framework",
                 "RMSD (Å)": round(sup.rms, 4), "CA": len(fw), "변이": len(muts(fw)), "변이 목록": "-"})
    for nm, (lo, hi) in spans.items():
        idx = [i for i in range(lo - 1, hi) if i in pair]
        m = muts(idx)
        rows.append({"부위": f"{chain}-{nm}", "정렬 기준": "이 체인 framework",
                     "RMSD (Å)": round(rmsd(idx), 4), "CA": len(idx),
                     "변이": len(m), "변이 목록": " ".join(m) or "-"})
    both = sorted(pair)
    s2 = Superimposer()
    s2.set_atoms([P[i]["CA"] for i in both], [Hm[pair[i]]["CA"] for i in both])
    rows.append({"부위": f"V{chain} 전체", "정렬 기준": "자체 CA 정렬",
                 "RMSD (Å)": round(s2.rms, 4), "CA": len(both), "변이": len(muts(both)), "변이 목록": "-"})
    head[chain] = {"fw": (round(sup.rms, 4), len(fw)), "whole": (round(s2.rms, 4), len(both))}
    if len(pair) < len(a):
        notes.append(f"V{chain} — {len(a)} aa 중 {len(pair)} 자리만 짝지어졌어요"
                     f" (짝 없는 parental 자리 {[i + 1 for i in range(len(a)) if i not in pair]}"
                     f" · humanized 자리 {[j + 1 for j in range(len(b)) if j not in set(pair.values())]}).")

tab = pd.DataFrame(rows)
tab.to_csv(MY / "cdr_rmsd_all.csv", index=False)
display(tab)
for s in notes:
    print(s)
print("→", MY / "cdr_rmsd_all.csv")

In [ ]:
h3 = tab[tab["부위"] == "H-CDR3"].iloc[0]
h3v, (fwv, fwn) = float(h3["RMSD (Å)"]), head["H"]["fw"]
whv, whn = head["H"]["whole"]

res = pd.DataFrame([
    {"metric": "framework_fit_rmsd", "value_angstrom": fwv, "n_atoms": fwn},
    {"metric": "cdr_h3_rmsd_after_framework_alignment", "value_angstrom": h3v, "n_atoms": int(h3["CA"])},
    {"metric": "whole_vh_rmsd_ca_aligned", "value_angstrom": whv, "n_atoms": whn},
])
res.to_csv(MY / "cdr_h3_rmsd_summary.csv", index=False)   # Ch.10 이 이 파일의 cdr_h3 행을 읽어요
display(res)

cdrs = tab[tab["부위"].str.contains("-CDR")]
over = cdrs[cdrs["RMSD (Å)"] >= 1.0]
worst = cdrs.loc[cdrs["RMSD (Å)"].idxmax()]
most  = cdrs.loc[cdrs["변이"].idxmax()]
print(f"CDR-H3 {h3v:.4f} Å 는 VH framework {fwv:.4f} Å 의 {h3v / fwv:.1f}배예요 — 그래도 절대값은 1.0 Å 아래예요.")
print(f"관찰 — CDR-H3 에 변이 {int(h3['변이'])}개({h3['변이 목록']})가 들어갔는데도 backbone 은 {h3v:.4f} Å 만 움직였어요.")
print("1.0 Å 이상 움직인 CDR —",
      ", ".join(f"{r['부위']} {r['RMSD (Å)']:.4f} Å(변이 {int(r['변이'])}개)" for _, r in over.iterrows()) or "없음")
print(f"가장 크게 움직인 CDR 은 {worst['부위']} ({worst['RMSD (Å)']:.4f} Å · 변이 {int(worst['변이'])}개), "
      f"변이가 가장 많은 CDR 은 {most['부위']} (변이 {int(most['변이'])}개 · {most['RMSD (Å)']:.4f} Å) 이에요.")
print("판정 — 변이 개수 순서와 움직인 정도의 순서가 달라요. 서열만 보고 '몇 개 안 바꿨으니 안전'이라 말할 수 없다는 뜻이라,")
print("       이렇게 접어서 재는 단계가 따로 필요해요. 1.0 Å 을 넘긴 CDR 은 backmutation 검토 대상으로 올려요.")

## 4) 겹쳐 보기 — 0.5406 Å 이 어디서 나왔는지 3D 로 (본문 8.2)

표의 숫자만으로는 **어디가** 움직였는지 알 수 없어요. 그래서 3절이 쓴 **VH framework 정렬을 그대로 재사용해** 두 구조를 겹쳐 놓고 봐요. 여기서 정렬을 새로 하면 안 돼요 — 기준틀이 달라지면 그림과 표가 서로 다른 이야기를 하게 되거든요.

- **파랑 = parental · 주황 = sapiens_humanized**(3절 `Superimposer` 로 옮겨 놓은 좌표)
- **진하고 굵은 구간 = CDR-H3**. 라벨 붙은 굵은 stick = 3절이 CDR-H3 안에서 찾아낸 **변이 잔기**(번호를 적어 넣지 않고 계산 결과에서 가져와요)
- 뷰어는 드래그로 돌리고 휠로 확대해요. 첫 뷰어는 Fv 전체, 두 번째는 CDR-H3 확대예요.

In [ ]:
_ensure("py3Dmol gemmi")
from Bio.PDB import PDBIO

try:                       # 뷰어가 없어도 좌표 파일·판정은 그대로 만들어요(설치 실패로 절이 통째로 죽지 않게)
    import py3Dmol
    HAVE_3D = True
except Exception as e:
    HAVE_3D = False
    print("py3Dmol 을 못 불러왔어요:", type(e).__name__, str(e)[:120])
    print("→ 3D 뷰어만 건너뛰고 나머지는 그대로 진행해요. `pip install py3Dmol` 뒤 셀을 다시 실행하면 그림이 나와요.")

# (1) 좌표 — 3절의 VH framework Superimposer 를 humanized 구조 '전체' 에 그대로 적용해요.
#     CDR-H3 RMSD 를 잰 바로 그 기준틀이라, 눈으로 보는 벌어짐과 표의 숫자가 같은 정렬 위에 있어요.
par_st = parser.get_structure("parental", str(pdbs["parental"]))
hum_st = parser.get_structure("humanized", str(pdbs["sapiens_humanized"]))
ALIGN["H"]["sup"].apply(list(hum_st.get_atoms()))

io, ovl = PDBIO(), {}
for tag, st in (("parental", par_st), ("humanized_aligned", hum_st)):
    io.set_structure(st)
    p = MY / f"overlay_{tag}.pdb"
    io.save(str(p))
    ovl[tag] = p

def pdb_text(p):
    """3Dmol.js 는 PDB 문자열이 가장 안정적이라 gemmi 로 한 번 통과시켜요(없으면 원문 그대로)."""
    try:
        import gemmi
        return gemmi.read_structure(str(p)).make_pdb_string()
    except Exception:
        return pathlib.Path(p).read_text()

def origin(p):
    return "내 결과 (my_run/)" if MY.resolve() in pathlib.Path(p).resolve().parents else "레퍼런스 (data/)"

print("[구조 출처] parental          ←", pdbs["parental"], "·", origin(pdbs["parental"]))
print("[구조 출처] sapiens_humanized ←", pdbs["sapiens_humanized"], "·", origin(pdbs["sapiens_humanized"]))
print("[정렬]      VH framework", f"{ALIGN['H']['sup'].rms:.4f} Å ({len(ALIGN['H']['fw'])} CA) — 3절과 같은 변환")

# (2) 강조할 잔기 — 앞 절이 계산한 CDR 구간·변이 목록에서 가져와요(잔기 번호 하드코딩 없음)
h3_lo, h3_hi = CDR[("H", "CDR3")]
pair_h = ALIGN["H"]["pair"]
par_i  = list(range(h3_lo - 1, h3_hi))                    # parental 체인 안 0-based
hum_i  = [pair_h[i] for i in par_i if i in pair_h]

def resi(name, chain, idxs):
    """체인 안 0-based 인덱스 → 그 PDB 파일의 실제 residue 번호."""
    s = tabs[name]
    s = s[s.chain == chain]
    m = dict(zip(s["pos"].astype(int), s["resnum"].astype(int)))
    return [int(m[i + 1]) for i in idxs if (i + 1) in m]

h3_par, h3_hum = resi("parental", "H", par_i), resi("sapiens_humanized", "H", hum_i)

mut_txt  = str(tab[tab["부위"] == "H-CDR3"].iloc[0]["변이 목록"]).strip()
mut_list = [] if mut_txt in ("-", "", "nan") else mut_txt.split()
mut_i    = [int("".join(ch for ch in m if ch.isdigit())) - 1 for m in mut_list]
mut_par  = resi("parental", "H", mut_i)
mut_hum  = resi("sapiens_humanized", "H", [pair_h[i] for i in mut_i if i in pair_h])
assert h3_par and h3_hum, "CDR-H3 잔기 번호를 못 찾았어요 — CDR 표가 이 구조의 것인지 확인하세요."

PAR_C, HUM_C = "#5b84c4", "#e07b39"

def overlay(zoom_sel=None, w=760, h=520):
    v = py3Dmol.view(width=w, height=h)
    v.addModel(pdb_text(ovl["parental"]), "pdb")             # model 0 = parental
    v.addModel(pdb_text(ovl["humanized_aligned"]), "pdb")    # model 1 = humanized (정렬된 좌표)
    v.setStyle({"model": 0}, {"cartoon": {"color": PAR_C, "opacity": 0.85}})
    v.setStyle({"model": 1}, {"cartoon": {"color": HUM_C, "opacity": 0.85}})
    v.addStyle({"model": 0, "chain": "H", "resi": h3_par},    # CDR-H3 = 굵은 cartoon + stick
               {"cartoon": {"color": "#123a72", "thickness": 1.0},
                "stick": {"colorscheme": "blueCarbon", "radius": 0.14}})
    v.addStyle({"model": 1, "chain": "H", "resi": h3_hum},
               {"cartoon": {"color": "#a02c0f", "thickness": 1.0},
                "stick": {"colorscheme": "redCarbon", "radius": 0.14}})
    if mut_par:                                              # CDR-H3 안의 변이 = 더 굵은 stick + 라벨
        v.addStyle({"model": 0, "chain": "H", "resi": mut_par},
                   {"stick": {"colorscheme": "cyanCarbon", "radius": 0.30}})
        v.addStyle({"model": 1, "chain": "H", "resi": mut_hum},
                   {"stick": {"colorscheme": "magentaCarbon", "radius": 0.30}})
        v.addResLabels({"model": 1, "chain": "H", "resi": mut_hum},
                       {"fontSize": 11, "fontColor": "black",
                        "backgroundColor": "white", "backgroundOpacity": 0.75})
    v.setBackgroundColor("white")
    if zoom_sel:
        v.zoomTo(zoom_sel); v.zoom(0.7)
    else:
        v.zoomTo()
    return v

print(f"\nCDR-H3 residue — parental {h3_par[0]}~{h3_par[-1]} ({len(h3_par)}개) · humanized {h3_hum[0]}~{h3_hum[-1]} ({len(h3_hum)}개)")
print(f"CDR-H3 안 변이 {len(mut_list)}개 —", " · ".join(mut_list) or "없음")
print("→", ovl["parental"], "·", ovl["humanized_aligned"])
if HAVE_3D:
    overlay().show()                                             # ① Fv 전체
    overlay({"model": 0, "chain": "H", "resi": h3_par}).show()    # ② CDR-H3 확대

In [ ]:
print(f"판정 — framework 는 {fwv:.4f} Å 라 두 cartoon 이 사실상 포개져요. 눈에 띄게 벌어지는 곳은 굵게 칠한 CDR-H3 이고,")
print(f"       그 벌어짐이 표의 {h3v:.4f} Å 예요. VH 전체는 {whv:.4f} Å 인데, 이건 loop 의 변화가 나머지 잔기에 희석된 값이에요.")
print(f"       라벨 붙은 {len(mut_list)}개({' · '.join(mut_list) or '없음'})가 CDR-H3 에 들어간 변이예요 —")
print("       곁사슬은 바뀌었는데 backbone cartoon 은 거의 같은 자리에 있어요. 그래서 '변이가 있다 = 구조가 망가졌다' 가 아니에요.")
if HAVE_3D:
    print("       그림이 안 보이면 py3Dmol 이 아니라 브라우저 쪽 문제예요 — Colab 이면 셀을 한 번 다시 실행해 보세요.")

## 5) 레퍼런스 대조 (본문 8.2)

`data/cdr_h3_rmsd_summary.csv` 는 이 가이드를 만들 때 같은 방식으로 뽑은 산출물이에요. 다만 **metric 이름이 노트북과 조금 달라요**(`whole_vh_rmsd_ca_aligned` ↔ `whole_vh_rmsd_all_atoms_aligned`). 이름 전체가 아니라 **앞부분만** 맞춰서 비교해요.

In [ ]:
ref = pd.read_csv(REF / "cdr_h3_rmsd_summary.csv")
STEMS = ("framework_fit", "cdr_h3", "whole_vh")

def stem(m):
    return next((s for s in STEMS if str(m).startswith(s)), str(m))

cmp_ = (res.assign(stem=res.metric.map(stem))
           .merge(ref.assign(stem=ref.metric.map(stem)), on="stem", suffixes=("_my", "_ref")))
cmp_["차이 (Å)"] = (cmp_.value_angstrom_my - cmp_.value_angstrom_ref).round(4)
display(cmp_[["stem", "value_angstrom_my", "n_atoms_my", "metric_ref", "value_angstrom_ref", "n_atoms_ref", "차이 (Å)"]])

print("레퍼런스 실측 — framework 0.2707 Å (91 CA) · CDR-H3 0.5406 Å (13 CA) · VH 전체 0.3207 Å (120 CA)")
same = bool((cmp_["차이 (Å)"].abs() < 0.005).all() and (cmp_.n_atoms_my == cmp_.n_atoms_ref).all())
if same:
    print("판정 — 세 지표가 레퍼런스와 같아요. 실행을 건너뛴 사람도 같은 표에 도달하니 아래 결론은 어느 쪽으로 읽어도 같아요.")
else:
    print("판정 — 값이 어긋나요. 내가 접은 구조나 Ch.05 후보 서열이 레퍼런스와 다르면 정상이에요.")
    print("       CA 개수까지 다르면 CDR 표가 이 서열의 것인지 먼저 확인하세요.")

print(f"\nCh.10 으로 넘어가는 값은 CDR-H3 RMSD {h3v:.4f} Å **하나**예요 — 나머지 표는 그 값을 해석하기 위한 맥락이에요.")
print("Ch.10 은 이 파일의 cdr_h3 행만 읽어 가고, 구조를 접어 본 후보(parental·Sapiens)에만 값이 붙어요.")
print("1.0 Å 아래라는 건 '통과'가 아니라 '구조 축에서 경고가 없다' 는 뜻이에요 — 결합력은 구조 하나로 정해지지 않아요.")

## 이 랩에서 확인한 것

1. IgFold 로 VH+VL Fv 를 접었어요 — 실측 **7.0859초**(parental) · **11.9880초**(humanized). `do_refine=False`(PyRosetta 없음) · `do_renum=False`(VL C-말단 잔기가 IMGT 범위 밖) 가 필수예요.
2. 잔기별 예측 오차(B-factor)는 **CDR 이 framework 의 1.4~2.1배** — loop 라서 정상이고, 두 구조가 같은 패턴이면 서로 비교해도 됩니다.
3. framework 정렬 후 **CDR-H3 = 0.5406 Å**(framework 0.2707 Å · **2.0배**, VH 전체 0.3207 Å). CDR-H3 에 변이 **2개(L104D·Y109V)** 가 들어갔는데도 이만큼만 움직였어요.
4. 6개 CDR 중 가장 크게 움직인 건 **L-CDR3 (2.5792 Å)** 인데 변이는 **1개**(R98S)뿐이에요. 반대로 변이가 4개인 L-CDR1 은 **0.6678 Å**. **변이 개수로 구조 변화를 예측할 수 없어요.**
5. 경쇄는 **framework 정렬 오차 자체가 1.2748 Å**(89 CA)로 중쇄(0.2707 Å · 91 CA)보다 훨씬 커요. VL 쪽 CDR 값은 이 바닥 위에서 읽어야 하고, C-말단 꼬리처럼 흔들리는 구간이 정렬을 끌고 갔는지 함께 봐야 해요.
6. ABodyBuilder3·ImmuneBuilder·AntiFold 는 **이 환경에서 미실행** — 여기 수치는 전부 IgFold 하나에서 나왔어요.
7. 3D 로 겹쳐 보면 **framework 는 포개지고 CDR-H3 만 벌어져요** — 표의 `0.2707 vs 0.5406` 이 그림 그대로예요. 겹치는 좌표는 새로 정렬한 게 아니라 3절 `Superimposer` 를 재사용한 것이라, 그림과 표가 같은 기준틀 위에 있어요.
8. 산출물 — `my_run/{parental,sapiens_humanized}.pdb` · `*_prmsd.csv` · `cdr_rmsd_all.csv` · `cdr_h3_rmsd_summary.csv` · `08_prmsd.png` · `overlay_{parental,humanized_aligned}.pdb`.

다음 → **[Ch.09 — Developability](../09_developability/09_developability_lab.ipynb)**